<a href="https://colab.research.google.com/github/NhanChanDeo/VQA_Final_Thesis/blob/main/DATASETS%20UPLOAD/Step2_3(1)_Fetch_Extract_DocVQA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Archive & File Structure Inspection: `gamusa/docvqa-task1-spdocvqa`

This notebook is specifically configured for **Google Colab** to systematically audit the file hierarchy, internal archive structure, file counts, and subdirectory counts of [`gamusa/docvqa-task1-spdocvqa`](https://huggingface.co/datasets/gamusa/docvqa-task1-spdocvqa).

### Can we count files and subdirectories inside each archive without downloading 8GB+ of images?
- **ZIP Archives (`.zip`):** **YES, 100% possible in milliseconds without downloading the archive content!** ZIP files store their entire table of contents (Central Directory) at the very end of the file. Using HTTP Range Requests (`Range: bytes=...`), we can read just the last few kilobytes to enumerate all file names, extensions, directory paths, and compressed/uncompressed sizes.
- **TAR Archives (`.tar.gz`):** TAR files are sequential streams where headers are interleaved across the file and compressed with gzip. While small `.tar.gz` files (like `spdocvqa_ocr.tar.gz` ~60MB) can be quickly downloaded in ~2-3 seconds, the large 8GB images archive can either be streamed sequentially via gzip without saving the uncompressed images to disk, or downloaded via `aria2c` in Colab if full audit is desired.

### What this notebook investigates:
1. **Hub Tree & Git Revision Audit:** Inspect repository root, subdirectories, and commit IDs.
2. **Archive Inventory & Member Counting:** Count files, distinct subdirectories, and extensions in:
   - `annotations/spdocvqa_qas.zip` (~1.7 MB)
   - `imdb/spdocvqa_imdb.zip` (~304 MB)
   - `ocr/spdocvqa_ocr.tar.gz` (~61 MB)
   - `images/spdocvqa_images.tar.gz` (~8.0 GB)
3. **Schema & Content Deep-Dive:** Inspect QA fields, OCR bounding boxes, and IMDB metadata.

In [7]:
# 1. Setup tools and dependencies
!pip install -q --upgrade huggingface_hub pillow matplotlib pandas
!sudo apt-get install -y -qq aria2

import os
import io
import json
import tarfile
import zipfile
from pathlib import Path
from collections import Counter
import pandas as pd
from huggingface_hub import HfApi, hf_hub_download, list_repo_files

REPO_ID = "gamusa/docvqa-task1-spdocvqa"
api = HfApi()
print(f"Connected to Hugging Face dataset: https://huggingface.co/datasets/{REPO_ID}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.0/838.0 kB 14.5 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 5.0.1 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.1.1 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.1.1 which is incompatible.
tokenizers 0.23.2 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.1.1 which is incompatible.
gradio-client 2.7.0 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.1.1 which is incompatible.
transformers 5.17.0 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 2.1.1 which is incompatible.
gradio 6.27.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 2.1.1 which is incompatible.
debconf: unable to initialize frontend: Dialog
debconf

In [8]:
# 2. Inspect Root Directory Tree and Remote Archives
repo_files = list_repo_files(repo_id=REPO_ID, repo_type="dataset")
print(f"Total items listed on repo: {len(repo_files)}\n")

archive_files = [f for f in repo_files if f.endswith(('.zip', '.tar.gz', '.tar'))]
other_files = [f for f in repo_files if f not in archive_files]

print("=== Primary Dataset Archives ===")
for a in archive_files:
    print(f"  • {a}")

print("\n=== Metadata & Config Files ===")
for o in other_files:
    print(f"  • {o}")

Total items listed on repo: 6

=== Primary Dataset Archives ===
  • annotations/spdocvqa_qas.zip
  • images/spdocvqa_images.tar.gz
  • imdb/spdocvqa_imdb.zip
  • ocr/spdocvqa_ocr.tar.gz

=== Metadata & Config Files ===
  • .gitattributes
  • README.md


In [9]:
# 3. Audit ZIP Archives: spdocvqa_qas.zip & spdocvqa_imdb.zip
# Analyzes file counts, subdirectories, and size statistics

def analyze_zip_archive(repo_id, filename):
    print(f"\n{'='*60}")
    print(f"Auditing ZIP: {filename}")
    print(f"{'='*60}")
    local_path = hf_hub_download(repo_id=repo_id, repo_type="dataset", filename=filename)
    
    with zipfile.ZipFile(local_path, 'r') as z:
        infolist = z.infolist()
        files = [i for i in infolist if not i.is_dir()]
        dirs = {Path(i.filename).parent for i in infolist if str(Path(i.filename).parent) != '.'}
        
        ext_counts = Counter(Path(f.filename).suffix for f in files)
        total_uncompressed = sum(f.file_size for f in files) / (1024 * 1024)
        total_compressed = sum(f.compress_size for f in files) / (1024 * 1024)
        
        print(f"✓ Total Files: {len(files)}")
        print(f"✓ Total Subdirectories: {len(dirs)}")
        if dirs:
            print(f"  Subdirectory list: {[str(d) for d in sorted(dirs)]}")
        print(f"✓ File Extensions Breakdown: {dict(ext_counts)}")
        print(f"✓ Archive Size: {total_compressed:.2f} MB (Uncompressed: {total_uncompressed:.2f} MB)")
        print("\nFirst 5 file entries:")
        for f in files[:5]:
            print(f"   - {f.filename} ({f.file_size / 1024:.1f} KB)")
    
    # Clean up local zip file from cache if space is critical
    return local_path

# Audit annotations (~1.7 MB - instantaneous)
qas_path = analyze_zip_archive(REPO_ID, "annotations/spdocvqa_qas.zip")

# Audit IMDBs (~304 MB - downloads in seconds)
imdb_path = analyze_zip_archive(REPO_ID, "imdb/spdocvqa_imdb.zip")


Auditing ZIP: annotations/spdocvqa_qas.zip
✓ Total Files: 3
✓ Total Subdirectories: 0
✓ File Extensions Breakdown: {'.json': 3}
✓ Archive Size: 1.63 MB (Uncompressed: 13.65 MB)

First 5 file entries:
   - test_v1.0.json (1123.0 KB)
   - train_v1.0_withQT.json (11294.0 KB)
   - val_v1.0_withQT.json (1556.3 KB)

Auditing ZIP: imdb/spdocvqa_imdb.zip
✓ Total Files: 3
✓ Total Subdirectories: 0
✓ File Extensions Breakdown: {'.npy': 3}
✓ Archive Size: 303.01 MB (Uncompressed: 941.45 MB)

First 5 file entries:
   - imdb_test.npy (106194.3 KB)
   - imdb_train.npy (747595.8 KB)
   - imdb_val.npy (110255.8 KB)


In [10]:
# 4. Audit OCR TAR Archive: ocr/spdocvqa_ocr.tar.gz (~61 MB)
print(f"\n{'='*60}")
print("Auditing TAR: ocr/spdocvqa_ocr.tar.gz (~61 MB)")
print(f"{'='*60}")

ocr_path = hf_hub_download(repo_id=REPO_ID, repo_type="dataset", filename="ocr/spdocvqa_ocr.tar.gz")

with tarfile.open(ocr_path, "r:gz") as tar:
    members = tar.getmembers()
    files = [m for m in members if m.isfile()]
    dirs = {Path(m.name).parent for m in members if str(Path(m.name).parent) != '.'}
    ext_counts = Counter(Path(f.name).suffix for f in files)
    total_uncompressed = sum(f.size for f in files) / (1024 * 1024)
    
    print(f"✓ Total Files: {len(files)}")
    print(f"✓ Distinct Subdirectories: {len(dirs)}")
    if len(dirs) <= 10:
        print(f"  Subdirectory list: {[str(d) for d in sorted(dirs)]}")
    else:
        print(f"  First 10 subdirectories: {[str(d) for d in sorted(dirs)[:10]]}...")
    print(f"✓ File Extensions Breakdown: {dict(ext_counts)}")
    print(f"✓ Total Uncompressed Size: {total_uncompressed:.2f} MB")
    print("\nFirst 5 OCR entries:")
    for f in files[:5]:
        print(f"   - {f.name} ({f.size / 1024:.1f} KB)")


Auditing TAR: ocr/spdocvqa_ocr.tar.gz (~61 MB)
✓ Total Files: 12767
✓ Distinct Subdirectories: 0
  Subdirectory list: []
✓ File Extensions Breakdown: {'.json': 12767}
✓ Total Uncompressed Size: 249.88 MB

First 5 OCR entries:
   - ffbf0023_4.json (48.8 KB)
   - ffbf0023_6.json (49.3 KB)
   - ffbf0227_1.json (9.4 KB)
   - ffbg0227_2.json (13.0 KB)
   - ffbg0227_3.json (10.9 KB)


In [11]:
# 5. Audit Document Images Archive: images/spdocvqa_images.tar.gz (~8.0 GB)
# Why we need to download or stream:
# TAR.GZ files do not store a global index header at the end of the file.
# Below, we stream-count members using tarfile sequentially so we do NOT extract 12,000 files to disk.

AUDIT_LARGE_IMAGES_TAR = True  # Set to False if you want to skip downloading the 8GB file

if AUDIT_LARGE_IMAGES_TAR:
    print(f"\n{'='*60}")
    print("Auditing TAR: images/spdocvqa_images.tar.gz (~8.0 GB)")
    print("Downloading via fast aria2c...")
    print(f"{'='*60}")
    
    img_url = f"https://huggingface.co/datasets/{REPO_ID}/resolve/main/images/spdocvqa_images.tar.gz"
    !aria2c -x 8 -s 8 -k 1M "{img_url}" -o spdocvqa_images.tar.gz
    
    print("Scanning tar member headers without extracting to disk...")
    with tarfile.open("spdocvqa_images.tar.gz", "r:gz") as tar:
        img_count = 0
        dirs = set()
        sample_files = []
        exts = Counter()
        
        for member in tar:
            if member.isfile():
                img_count += 1
                p = Path(member.name)
                exts[p.suffix] += 1
                if str(p.parent) != '.':
                    dirs.add(p.parent)
                if len(sample_files) < 5:
                    sample_files.append((member.name, member.size))
        
        print(f"✓ Total Document Images: {img_count}")
        print(f"✓ Total Distinct Subdirectories: {len(dirs)}")
        if dirs:
            print(f"  Subdirectories: {[str(d) for d in sorted(dirs)]}")
        print(f"✓ Image Extensions Breakdown: {dict(exts)}")
        print("\nFirst 5 Image Entries:")
        for name, size in sample_files:
            print(f"   - {name} ({size / 1024:.1f} KB)")
            
    # Clean up archive to keep Colab disk pristine
    !rm -f spdocvqa_images.tar.gz
    print("✓ Cleaned up spdocvqa_images.tar.gz from disk")
else:
    print("Skipping 8GB images download. Known official count: ~12,767 document images.")


Auditing TAR: images/spdocvqa_images.tar.gz (~8.0 GB)

10/01 14:01:20 [NOTICE] Downloading 1 item(s)

10/01 14:01:20 [NOTICE] CUID#7 - Redirecting to https://us.gcp.cdn.hf.co/xet-bridge-us/6abdd8ac8882209890d5809a/9ebe0d8048746da12d0bea31c3bf7b469f0bbb395f6bfb4ab69bd3eb81359023?X-Xet-Cas-Uid=public&user_id=public&xip=j2u5Zg5gSzo&response-content-type=application%2Fgzip&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27spdocvqa_images.tar.gz%3B+filename%3D%22spdocvqa_images.tar.gz%22%3B&Expires=1790866880&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNmFiZGQ4YWM4ODgyMjA5ODkwZDU4MDlhLzllYmUwZDgwNDg3NDZkYTEyZDBiZWEzMWMzYmY3YjQ2OWYwYmJiMzk1ZjZiZmI0YWI2OWJkM2ViODEzNTkwMjNcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZ1c2VyX2lkPXB1YmxpYyZ4aXA9ajJ1NVpnNWdTem8mcmVzcG9uc2UtY29udGVudC10eXBlPSomcmVzcG9uc2UtY29udGVudC1kaXNwb3NpdGlvbj0qIiwiQ29uZGl0aW9uIjp7IkRhdGVMZXNzVGhhbiI6eyJFcG9jaFRpbWUiOjE3OTA4NjY4ODB9fX1dfQ__&Signature=MEQCIDAJ9Yc7JgBLmj5xypBODJX

In [12]:
# 6. Schema & Content Inspection (QA, OCR, IMDB)
print("=== 1. Sample Annotation Record (QA Pair) ===")
with zipfile.ZipFile(qas_path, 'r') as z:
    # Load train split
    train_json = next(name for name in z.namelist() if 'train' in name and name.endswith('.json'))
    print(f"Opening: {train_json}")
    with z.open(train_json) as f:
        qas_data = json.load(f)
        records = qas_data.get('data', qas_data)
        print(f"Total QA records in {train_json}: {len(records)}")
        print("\nSample QA Record:")
        print(json.dumps(records[0], indent=2))

print("\n=== 2. Sample OCR Record ===")
with tarfile.open(ocr_path, "r:gz") as tar:
    sample_m = next(m for m in tar.getmembers() if m.isfile() and m.name.endswith('.json'))
    print(f"Opening: {sample_m.name}")
    ocr_sample = json.load(tar.extractfile(sample_m))
    print(json.dumps(ocr_sample, indent=2)[:700], "... [truncated]")

print("\n=== 3. Sample IMDB Entry ===")
with zipfile.ZipFile(imdb_path, 'r') as z:
    imdb_file = next(name for name in z.namelist() if not name.endswith('/'))
    print(f"Opening: {imdb_file}")
    with z.open(imdb_file) as f:
        preview = f.read(1024)
        print("First 1024 bytes preview:\n", preview.decode('utf-8', errors='replace')[:500])

=== 1. Sample Annotation Record (QA Pair) ===
Opening: train_v1.0_withQT.json
Total QA records in train_v1.0_withQT.json: 39463

Sample QA Record:
{
  "questionId": 337,
  "question": "what is the date mentioned in this letter?",
  "question_types": [
    "handwritten",
    "form"
  ],
  "image": "documents/xnbl0037_1.png",
  "docId": 279,
  "ucsf_document_id": "xnbl0037",
  "ucsf_document_page_no": "1",
  "answers": [
    "1/8/93"
  ],
  "data_split": "train"
}

=== 2. Sample OCR Record ===
Opening: ffbf0023_4.json
{
  "status": "Succeeded",
  "recognitionResults": [
    {
      "page": 1,
      "clockwiseOrientation": 359.96,
      "width": 1692,
      "height": 2245,
      "unit": "pixel",
      "lines": [
        {
          "boundingBox": [
            622,
            138,
            1137,
            136,
            1138,
            167,
            622,
            168
          ],
          "text": "R. J. REYNOLDS TOBACCO COMPANY",
          "words": [
            {
       

## 7. Extract Selected Archives (`annotations/` and `imdb/`) & Sync to Hugging Face

As determined in our analysis:
- **`annotations/spdocvqa_qas.zip` (~1.7 MB, ~3 JSON files):** Extracting enables the native Hugging Face **Dataset Viewer** preview on the web UI and fast split loading.
- **`imdb/spdocvqa_imdb.zip` (~304 MB, ~3 JSONL files):** Extracting provides direct access to index files without tens of thousands of file entries.
- **`ocr/` and `images/`:** Left compressed as `.tar.gz` archives to prevent Git tree bloat (avoiding 25,000+ tiny files and multi-commit splits).

The cell below:
1. Downloads `annotations/spdocvqa_qas.zip` and `imdb/spdocvqa_imdb.zip`.
2. Decompresses both archives into `annotations/` and `imdb/`.
3. Removes the local `.zip` files.
4. (Optional) Syncs the extracted files to Hugging Face using `HfApi.upload_folder` with `delete_patterns=['*.zip']` to replace the remote archives with the clean uncompressed files.

In [13]:
# 7.1 Extract annotations and imdb archives
import zipfile
import os
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download

# Working directories
EXTRACT_DIR = Path("./docvqa_extracted_selected")
ANNOTATIONS_DIR = EXTRACT_DIR / "annotations"
IMDB_DIR = EXTRACT_DIR / "imdb"

ANNOTATIONS_DIR.mkdir(parents=True, exist_ok=True)
IMDB_DIR.mkdir(parents=True, exist_ok=True)

# --- 1. Download and Extract Annotations (QAs) ---
print(">>> Downloading annotations/spdocvqa_qas.zip...")
qas_zip_path = hf_hub_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    filename="annotations/spdocvqa_qas.zip"
)

print(">>> Extracting annotations into annotations/...")
with zipfile.ZipFile(qas_zip_path, 'r') as z:
    z.extractall(ANNOTATIONS_DIR)
print("✓ Annotations extracted successfully. Extracted files:")
for f in ANNOTATIONS_DIR.iterdir():
    print(f"   - {f.name} ({f.stat().st_size / 1024:.1f} KB)")

# --- 2. Download and Extract IMDB (Image Database) ---
print("\n>>> Downloading imdb/spdocvqa_imdb.zip...")
imdb_zip_path = hf_hub_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    filename="imdb/spdocvqa_imdb.zip"
)

print(">>> Extracting IMDB files into imdb/...")
with zipfile.ZipFile(imdb_zip_path, 'r') as z:
    z.extractall(IMDB_DIR)
print("✓ IMDB extracted successfully. Extracted files:")
for f in IMDB_DIR.iterdir():
    print(f"   - {f.name} ({f.stat().st_size / (1024*1024):.1f} MB)")

print("\nAll targeted archives extracted with preserved directory layout!")

>>> Downloading annotations/spdocvqa_qas.zip...
>>> Extracting annotations into annotations/...
✓ Annotations extracted successfully. Extracted files:
   - train_v1.0_withQT.json (11294.0 KB)
   - test_v1.0.json (1123.0 KB)
   - val_v1.0_withQT.json (1556.3 KB)

>>> Downloading imdb/spdocvqa_imdb.zip...
>>> Extracting IMDB files into imdb/...
✓ IMDB extracted successfully. Extracted files:
   - imdb_train.npy (730.1 MB)
   - imdb_val.npy (107.7 MB)
   - imdb_test.npy (103.7 MB)

All targeted archives extracted with preserved directory layout!


In [15]:

# 7.2 (Optional) Push uncompressed annotations & imdb to Hugging Face Hub
# Uncomment the code below to sync changes back to your repository


# Authenticate with write token
from huggingface_hub import login
login(token=HF_TOKEN)

api = HfApi()

# Sync annotations/ folder (replaces spdocvqa_qas.zip with extracted JSON files)
print(">>> Syncing extracted annotations to Hugging Face Hub...")
api.upload_folder(
    folder_path=str(ANNOTATIONS_DIR),
    path_in_repo="annotations",
    repo_id=REPO_ID,
    repo_type="dataset",
    commit_message="Extract annotations: replace spdocvqa_qas.zip with native JSON splits",
    delete_patterns=["*.zip"]
)

# Sync imdb/ folder (replaces spdocvqa_imdb.zip with extracted JSONL files)
print(">>> Syncing extracted imdb to Hugging Face Hub...")
api.upload_folder(
    folder_path=str(IMDB_DIR),
    path_in_repo="imdb",
    repo_id=REPO_ID,
    repo_type="dataset",
    commit_message="Extract imdb: replace spdocvqa_imdb.zip with native JSONL splits",
    delete_patterns=["*.zip"]
)

print("✓ Selected folders updated on Hugging Face Hub!")


NameError: name 'HF_TOKEN' is not defined